# Part 1: Tensors, Gradients, and GPUs

Welcome to the world of Deep Learning with **PyTorch**! 

In the previous lesson, we set up our environment. Now, we will learn about the fundamental building block of deep learning: the **Tensor**.

**Goals for this lesson:**
1.  Understand what a **Tensor** is and how to create one.
2.  Learn how to use the **GPU** to speed up calculations.
3.  Understand **Broadcasting** (a key concept from the slides).
4.  Learn how **Autograd** automatically calculates gradients for us.

---

## 1. What is PyTorch?

PyTorch is the most popular deep learning framework for research. It provides two main things:
1.  **N-dimensional Tensors** (like NumPy arrays) that can run on **GPUs**.
2.  **Automatic Differentiation** (Autograd) for training neural networks.

Let's check our version:

In [1]:
import torch
import numpy as np

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

PyTorch version: 2.10.0+cpu
CUDA available: False


## 2. The Tensor

A tensor is just a fancy name for a multi-dimensional array. 
* **Scalar:** 0-D tensor (a single number)
* **Vector:** 1-D tensor
* **Matrix:** 2-D tensor
* **Image:** 3-D tensor (Height, Width, Color Channels)

Let's look at ways to create them.

In [2]:
# 1. From a Python List
t_list = torch.tensor([[1, 2, 3], [4, 5, 6]])
print(f"From list:\n{t_list}")
print(f"Shape: {t_list.shape}")

# 2. Zeros and Ones (useful for initializing weights)
t_zeros = torch.zeros((2, 3))
t_ones = torch.ones((2, 3))
print(f"\nZeros:\n{t_zeros}")

# 3. Random values (Essential for neural networks!)
t_rand = torch.rand((3, 3))
print(f"\nRandom (0 to 1):\n{t_rand}")

From list:
tensor([[1, 2, 3],
        [4, 5, 6]])
Shape: torch.Size([2, 3])

Zeros:
tensor([[0., 0., 0.],
        [0., 0., 0.]])

Random (0 to 1):
tensor([[0.8027, 0.6130, 0.4552],
        [0.4995, 0.7130, 0.9739],
        [0.4755, 0.1382, 0.2006]])


## 3. PyTorch vs NumPy (Memory Magic)

PyTorch is designed to work perfectly with NumPy. 
**Crucial Concept:** If you convert between them on the CPU, they **share memory**. Changing one changes the other!

In [3]:
np_arr = np.array([1, 2, 3])
torch_tensor = torch.from_numpy(np_arr)

print(f"Original NumPy: {np_arr}")
print(f"PyTorch Tensor: {torch_tensor}")

print("\n--- Modifying NumPy array ---")
np_arr[0] = 100
print(f"New PyTorch Tensor: {torch_tensor}  <-- It changed!")

Original NumPy: [1 2 3]
PyTorch Tensor: tensor([1, 2, 3])

--- Modifying NumPy array ---
New PyTorch Tensor: tensor([100,   2,   3])  <-- It changed!


## 4. Using the GPU (The Speedup)

This is why we use PyTorch. NumPy only runs on the CPU. PyTorch tensors can live on the GPU.

To move a tensor, we use `.to(device)`.

In [4]:
# Determine the device
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("✅ Using NVIDIA GPU (CUDA)")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("✅ Using Apple Metal (MPS)")
else:
    device = torch.device("cpu")
    print("⚠️ Using CPU")

# Create a tensor on CPU (default)
x = torch.tensor([10, 20, 30])
print(f"Location: {x.device}")

# Move to GPU
x_gpu = x.to(device)
print(f"New Location: {x_gpu.device}")

# Calculate on GPU
result = x_gpu * 2
print(f"Result on GPU: {result}")

# Move back to CPU (e.g., for plotting with Matplotlib)
x_back = result.cpu()
print(f"Back on CPU: {x_back.device}")

⚠️ Using CPU
Location: cpu
New Location: cpu
Result on GPU: tensor([20, 40, 60])
Back on CPU: cpu


## 5. Tensor Operations & Broadcasting

**Broadcasting** is a magic rule that allows you to do math with tensors of different shapes. PyTorch automatically "stretches" the smaller tensor to match the larger one.

Example from slides: Adding a vector `(2)` to a matrix `(2, 4)`.

In [5]:
# Matrix (2 rows, 4 columns)
A = torch.tensor([[1, 2, 3, 4], 
                  [5, 6, 7, 8]])

# Vector (2 rows, 1 column)
b = torch.tensor([[10], 
                  [20]])

print(f"Shape of A: {A.shape}")
print(f"Shape of b: {b.shape}")

# Adding them works! 'b' is copied across all columns of 'A'
C = A + b
print(f"\nResult (Broadcasting):\n{C}")

Shape of A: torch.Size([2, 4])
Shape of b: torch.Size([2, 1])

Result (Broadcasting):
tensor([[11, 12, 13, 14],
        [25, 26, 27, 28]])


## 6. Reshaping: Flattening

Neural networks often require changing shapes (e.g., turning a 2D image into a 1D vector). 
We use `.view()` or `.reshape()`.

In [30]:
batch_images = torch.rand(10, 28, 28) # 10 images, 28x28 pixels
print(f"Original shape: {batch_images.shape}")

# Flatten: Keep batch (10), stretch the rest (28*28 = 784)
flat = batch_images.reshape(10, -1) 
print(f"Flattened shape: {flat.shape}")

Original shape: torch.Size([10, 28, 28])
Flattened shape: torch.Size([10, 784])


## 7. Autograd: The Engine of Deep Learning

This is the most important part. PyTorch knows how to calculate derivatives (gradients) automatically. This is how neural networks learn from their errors.

We use `requires_grad=True` to tell PyTorch: "Please track this variable."

In [7]:
# 1. Create a variable we want to optimize (e.g., a weight)
w = torch.tensor(4.0, requires_grad=True)

# 2. Define a function (loss)
# Let's say Loss = 3 * w^2
# The derivative d(Loss)/dw is 6 * w
loss = 3 * w**2

print(f"Current Loss: {loss}")

# 3. Backward Pass (Calculate Gradients)
loss.backward()

# 4. Check the gradient
# At w=4, gradient should be 6 * 4 = 24
print(f"Gradient (dL/dw): {w.grad}")

# 5. Important: Reset gradients before next step!
w.grad.zero_()

Current Loss: 48.0
Gradient (dL/dw): 24.0


tensor(0.)

## 8. Practice Exercises

1.  Create a random tensor of shape `(5, 5)`. Move it to the GPU (if available).
2.  Create two tensors: `A` of shape `(3, 1)` and `B` of shape `(1, 3)`. Add them together. What is the shape of the result? (Broadcasting test).
3.  Calculate the gradient of `y = x^3 + 5` at `x=2`. (Theoretical answer: $3x^2 = 12$).

In [8]:
# --- Write your solutions here ---
ten = torch.rand((5, 5))

In [9]:
ten

tensor([[0.4331, 0.2419, 0.1002, 0.6311, 0.9260],
        [0.1548, 0.6772, 0.4339, 0.3826, 0.4023],
        [0.3804, 0.7163, 0.2000, 0.2261, 0.4588],
        [0.1799, 0.8034, 0.5592, 0.0365, 0.7043],
        [0.7023, 0.5848, 0.8138, 0.9570, 0.7535]])

In [10]:
tenA = torch.tensor([1, 2, 3]).view(3,1)
tenA

tensor([[1],
        [2],
        [3]])

In [11]:
tenB = torch.tensor([4, 5, 6]).view(1,3)
tenB

tensor([[4, 5, 6]])

In [12]:
tenA + tenB

tensor([[5, 6, 7],
        [6, 7, 8],
        [7, 8, 9]])

# For autograd we track the variable/weight

In [36]:
x = torch.tensor(2.0, requires_grad=True)
y = x**3 + 5
print(f"y = {y}")

y.backward()
print(f"dy/dx_(for x=2) = {x.grad}")

x.grad.zero_()

y = 13.0
dy/dx_(for x=2) = 12.0


tensor(0.)